# Fine-tuning LLM for Ransomware Detection

This notebook demonstrates how to fine-tune a small LLM to detect ransomware from telemetry data.

## 1. Setup and Dependencies

In [ ]:
# Install dependencies (run once)
!pip install -q transformers datasets accelerate peft bitsandbytes torch

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Generate Training Data

In [ ]:
def create_training_examples(num_examples=50):
    """Create simple training examples."""
    examples = []
    
    for i in range(num_examples):
        is_ransomware = i % 3 != 0
        
        if is_ransomware:
            prompt = """<|telemetry|>
Process: powershell.exe -enc SGVsbG8=
Process: vssadmin.exe delete shadows
File: Multiple files .encrypted
Network: 185.220.101.45:443

Analyze for ransomware:"""
            
            completion = """RANSOMWARE DETECTED
State: encryption_active
Risk: CRITICAL
Evidence: Shadow deletion, file encryption
Action: Isolate immediately"""
        else:
            prompt = """<|telemetry|>
Process: chrome.exe
File: document.docx accessed
Network: microsoft.com

Analyze for ransomware:"""
            
            completion = """BENIGN
State: normal_activity
Risk: LOW
Evidence: Standard applications
Action: Continue monitoring"""
        
        examples.append({
            "text": f"{prompt}\n{completion}"
        })
    
    return examples

# Generate examples
train_examples = create_training_examples(100)
eval_examples = create_training_examples(20)

print(f"Generated {len(train_examples)} training examples")
print(f"Generated {len(eval_examples)} evaluation examples")
print("\nExample:")
print(train_examples[0]['text'][:200] + "...")

## 3. Prepare Dataset

In [ ]:
# Create datasets
train_dataset = Dataset.from_list(train_examples)
eval_dataset = Dataset.from_list(eval_examples)

print(f"Train dataset: {train_dataset}")
print(f"Eval dataset: {eval_dataset}")

## 4. Load Model with QLoRA

In [ ]:
# Model selection based on GPU
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # 4GB VRAM
# MODEL_NAME = "microsoft/phi-2"  # 8GB VRAM
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"  # 12GB+ VRAM

print(f"Loading model: {MODEL_NAME}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model with 4-bit quantization
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded!")

## 5. Configure LoRA

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # Target attention layers
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

print("Datasets tokenized!")

## 7. Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir="./ransomware_detector",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    report_to="none"
)

# Create trainer
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("Trainer configured!")

## 8. Train the Model

In [ ]:
# Start training
print("Starting training...")
trainer.train()

# Save model
trainer.save_model("./ransomware_detector/final")
print("Training complete! Model saved.")

## 9. Test the Fine-tuned Model

In [ ]:
def test_model(prompt):
    """Test the model with a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Test with ransomware scenario
test_prompt_ransomware = """<|telemetry|>
Process: powershell.exe -enc [encoded]
Process: vssadmin delete shadows
File: encryption detected

Analyze for ransomware:"""

print("Test 1: Ransomware Scenario")
print("Input:", test_prompt_ransomware)
print("\nModel Response:")
print(test_model(test_prompt_ransomware))

# Test with benign scenario
test_prompt_benign = """<|telemetry|>
Process: notepad.exe
File: readme.txt opened

Analyze for ransomware:"""

print("\n" + "="*50)
print("Test 2: Benign Scenario")
print("Input:", test_prompt_benign)
print("\nModel Response:")
print(test_model(test_prompt_benign))

## 10. Evaluation Metrics

In [ ]:
def evaluate_accuracy(test_prompts):
    """Evaluate model accuracy on test prompts."""
    correct = 0
    total = 0
    
    for prompt, expected in test_prompts:
        response = test_model(prompt).lower()
        
        if expected == "ransomware" and "ransomware" in response:
            correct += 1
        elif expected == "benign" and "benign" in response:
            correct += 1
        
        total += 1
    
    accuracy = correct / total
    print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")
    return accuracy

# Create test set
test_prompts = [
    ("<|telemetry|>\nProcess: vssadmin delete\n\nAnalyze:", "ransomware"),
    ("<|telemetry|>\nProcess: chrome.exe\n\nAnalyze:", "benign"),
    ("<|telemetry|>\nFile: .encrypted files\n\nAnalyze:", "ransomware"),
    ("<|telemetry|>\nFile: document.pdf\n\nAnalyze:", "benign"),
]

evaluate_accuracy(test_prompts)

## Summary

This notebook demonstrates:
1. **Data Generation**: Creating training examples from telemetry patterns
2. **Model Loading**: Using QLoRA for efficient fine-tuning
3. **Training**: Fine-tuning with LoRA adapters
4. **Testing**: Evaluating on ransomware detection tasks

The model learns to:
- Identify ransomware patterns in telemetry
- Assess risk levels
- Recommend appropriate actions
- Explain its reasoning

Next steps:
- Use more sophisticated telemetry data
- Train on longer sequences
- Add memory/RAG for long-term patterns
- Implement policy head for action selection